<a href="https://colab.research.google.com/github/amanpoonia/SBeta_preparation/blob/main/results_fama_french.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
factors = pd.read_csv('FF_factor_values.csv', sep=',')

In [25]:
display(factors.head())
factors.info()

,Unnamed: 0,Mkt-RF,SMB,HML,RF
0,192607,2.89,-2.42,-2.75,0.22
1,192608,2.64,-1.44,4.13,0.25
2,192609,0.38,-1.36,0.05,0.23
3,192610,-3.27,-0.18,0.78,0.32
4,192611,2.54,-0.17,-0.48,0.31


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1202 entries, 0 to 1201
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  1202 non-null   int64  
 1   Mkt-RF      1202 non-null   float64
 2   SMB         1202 non-null   float64
 3   HML         1202 non-null   float64
 4   RF          1202 non-null   float64
dtypes: float64(4), int64(1)
memory usage: 47.1 KB


In [21]:
portfolios = pd.read_csv('25x25_3fac_portfolios.csv', sep=',')
portfolios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1202 entries, 0 to 1201
Data columns (total 26 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  1202 non-null   int64  
 1   SMALL LoBM  1202 non-null   float64
 2   ME1 BM2     1202 non-null   float64
 3   ME1 BM3     1202 non-null   float64
 4   ME1 BM4     1202 non-null   float64
 5   SMALL HiBM  1202 non-null   float64
 6   ME2 BM1     1202 non-null   float64
 7   ME2 BM2     1202 non-null   float64
 8   ME2 BM3     1202 non-null   float64
 9   ME2 BM4     1202 non-null   float64
 10  ME2 BM5     1202 non-null   float64
 11  ME3 BM1     1202 non-null   float64
 12  ME3 BM2     1202 non-null   float64
 13  ME3 BM3     1202 non-null   float64
 14  ME3 BM4     1202 non-null   float64
 15  ME3 BM5     1202 non-null   float64
 16  ME4 BM1     1202 non-null   float64
 17  ME4 BM2     1202 non-null   float64
 18  ME4 BM3     1202 non-null   float64
 19  ME4 BM4     1202 non-null  

To find the alpha for every portfolio (as well as the beta for all the portfolios)

we run regression on the factors as the independent variable and the portfolio return as the dependent variable.

**MODELS**
1) CAPM
2) 3 Factors: market risk + Size + Value
3) 5 Factors: 3 factors + profitability + investment

In [28]:
portfolio_name_list = portfolios.columns[1:].to_list()

#factors
X1 = sm.add_constant(factors[['Mkt-RF']])
X3 = sm.add_constant(factors[['Mkt-RF','SMB','HML']])

In [29]:
#Regressing the portfolio returns against the factors
#result matrix presents the ALPHA of the regression
alphas_1 = []
alphas_3 = []
for i in portfolio_name_list:
  res1 = sm.OLS(portfolios[i], X1).fit(cov_type = "HAC", cov_kwds={"maxlags": 6})
  res3 = sm.OLS(portfolios[i], X3).fit(cov_type = "HAC", cov_kwds={"maxlags": 6})

  #print(f"{i}, {res.params.const}")
  alphas_1.append(res1.params.const)
  alphas_3.append(res3.params.const)

alpha_matrix_1 = np.array(alphas_1).reshape(5,5)
alpha_matrix_3 = np.array(alphas_3).reshape(5,5)
display(alpha_matrix_1)
display(alpha_matrix_3)

array([[-0.26858939, -0.00399315,  0.31678309,  0.54347994,  0.6776565 ],
       [ 0.05651763,  0.36433468,  0.41973332,  0.469954  ,  0.55637139],
       [ 0.13997817,  0.40054083,  0.4075819 ,  0.46163728,  0.44163439],
       [ 0.24404268,  0.29473573,  0.36440927,  0.43643005,  0.34492723],
       [ 0.29481669,  0.27907961,  0.32918765,  0.18422501,  0.3751046 ]])

array([[-0.40534208, -0.10642238,  0.15118756,  0.35897996,  0.41354493],
       [ 0.07290012,  0.29961724,  0.30607365,  0.29529767,  0.30478274],
       [ 0.16516638,  0.36798884,  0.31121625,  0.30485381,  0.20639392],
       [ 0.31288642,  0.25888518,  0.27048225,  0.28733887,  0.09827166],
       [ 0.36624633,  0.27838626,  0.25345831,  0.02662789,  0.13452043]])

In the above result matrices each ij is comparable, the change in alpha shows whether the introduced factors are able to explain more.

In [32]:
#DISPLAYING THE RESULTS
size_labels = ['small','me2','me3','me4','big']
value_labels = ['lowbm_growth','bm2','bm3','bm4','highbm_value']

capm_alpha = pd.DataFrame(alpha_matrix_1, index=size_labels, columns=value_labels)
factor3_alpha = pd.DataFrame(alpha_matrix_3, index=size_labels, columns=value_labels)

display(capm_alpha)
display(factor3_alpha)

,lowbm_growth,bm2,bm3,bm4,highbm_value
small,-0.268589,-0.003993,0.316783,0.543480,0.677657
me2,0.056518,0.364335,0.419733,0.469954,0.556371
me3,0.139978,0.400541,0.407582,0.461637,0.441634
me4,0.244043,0.294736,0.364409,0.436430,0.344927
big,0.294817,0.279080,0.329188,0.184225,0.375105


,lowbm_growth,bm2,bm3,bm4,highbm_value
small,-0.405342,-0.106422,0.151188,0.358980,0.413545
me2,0.072900,0.299617,0.306074,0.295298,0.304783
me3,0.165166,0.367989,0.311216,0.304854,0.206394
me4,0.312886,0.258885,0.270482,0.287339,0.098272
big,0.366246,0.278386,0.253458,0.026628,0.134520


In [33]:
#--AI--
# Create a visually styled table using a Seaborn color palette gradient
cm = sns.light_palette("green", as_cmap=True)
styled_table_1 = capm_alpha.style.background_gradient(cmap=cm).format("{:.4f}")
styled_table_3 = factor3_alpha.style.background_gradient(cmap=cm).format("{:.4f}")
display(styled_table_1)
display(styled_table_3)

,lowbm_growth,bm2,bm3,bm4,highbm_value
small,-0.2686,-0.0040,0.3168,0.5435,0.6777
me2,0.0565,0.3643,0.4197,0.4700,0.5564
me3,0.1400,0.4005,0.4076,0.4616,0.4416
me4,0.2440,0.2947,0.3644,0.4364,0.3449
big,0.2948,0.2791,0.3292,0.1842,0.3751


,lowbm_growth,bm2,bm3,bm4,highbm_value
small,-0.4053,-0.1064,0.1512,0.3590,0.4135
me2,0.0729,0.2996,0.3061,0.2953,0.3048
me3,0.1652,0.3680,0.3112,0.3049,0.2064
me4,0.3129,0.2589,0.2705,0.2873,0.0983
big,0.3662,0.2784,0.2535,0.0266,0.1345


In [22]:
#DONOT RUN
# Single portfolio results
'''
X = sm.add_constant(factors[['Mkt-RF','SMB','HML']])
res = sm.OLS(portfolios['SMALL LoBM'], X).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
print(res.params, res.pvalues)
'''

const    -0.405342
Mkt-RF    1.269962
SMB       1.469714
HML       0.347309
dtype: float64 const     4.871027e-03
Mkt-RF    8.220223e-45
SMB       4.711548e-17
HML       1.252054e-01
dtype: float64


In [23]:


# Create a visually styled table using a Seaborn color palette gradient
cm = sns.light_palette("green", as_cmap=True)
styled_table = alpha_df.style.background_gradient(cmap=cm).format("{:.4f}")
display(styled_table)

,BM1 (Low),BM2,BM3,BM4,BM5 (High)
ME1 (Small),-0.4053,-0.1064,0.1512,0.3590,0.4135
ME2,0.0729,0.2996,0.3061,0.2953,0.3048
ME3,0.1652,0.3680,0.3112,0.3049,0.2064
ME4,0.3129,0.2589,0.2705,0.2873,0.0983
ME5 (Big),0.3662,0.2784,0.2535,0.0266,0.1345


In [ ]:
mean_alpha = np.array(a)